# LLM Dataset Creation Notebook

This notebook extracts text/code files from a local directory (e.g. `C:\Users\adams\.gemini\antigravity-ide` or any repository) and formats them into JSONL training files for pre-training, fine-tuning (SFT chat), or instruction tuning.

In [ ]:
import os
import json
import sys

# Target directory and output settings
TARGET_DIR = '.'  # Set to target folder, e.g., r'C:\Users\adams\.gemini\antigravity-ide'
OUTPUT_DIR = './output_datasets'
os.makedirs(OUTPUT_DIR, exist_ok=True)

IGNORE_DIRS = {".git", "node_modules", "__pycache__", ".venv", "venv", ".idea", ".vscode", "dist", "build"}
BINARY_EXTENSIONS = {
    ".exe", ".dll", ".so", ".dylib", ".pyc", ".pyd", ".png", ".jpg", ".jpeg",
    ".gif", ".ico", ".svg", ".zip", ".tar", ".gz", ".7z", ".pdf", ".db",
    ".sqlite", ".bin", ".dat", ".wof", ".woff", ".woff2", ".ttf", ".eot",
    ".mp3", ".mp4", ".wav", ".avi", ".mov", ".webm"
}

def is_binary(file_path: str) -> bool:
    try:
        with open(file_path, "rb") as f:
            return b"\0" in f.read(1024)
    except Exception:
        return True

def chunk_text(text: str, max_chars: int = 4000):
    if not max_chars or len(text) <= max_chars:
        return [text]
    return [text[i:i + max_chars] for i in range(0, len(text), max_chars)]

print("Helper functions initialized.")

In [ ]:
def extract_files(root_dir):
    root_path = os.path.abspath(root_dir)
    extracted = []
    for current_root, dirs, files in os.walk(root_path):
        dirs[:] = [d for d in dirs if d not in IGNORE_DIRS]
        for f_name in files:
            ext = os.path.splitext(f_name)[1].lower()
            if ext in BINARY_EXTENSIONS:
                continue
            full_path = os.path.join(current_root, f_name)
            rel_path = os.path.relpath(full_path, root_path)
            if is_binary(full_path):
                continue
            try:
                with open(full_path, "r", encoding="utf-8", errors="replace") as f:
                    content = f.read()
                    if content.strip():
                        extracted.append((rel_path, content))
            except Exception as e:
                print(f"Skipping {rel_path}: {e}")
    return extracted

files = extract_files(TARGET_DIR)
print(f"Found {len(files)} valid text/code files in {TARGET_DIR}")

In [ ]:
# Pre-training Dataset (text format)
pretrain_path = os.path.join(OUTPUT_DIR, "pretrain_dataset.jsonl")
pretrain_count = 0
with open(pretrain_path, "w", encoding="utf-8") as out:
    for rel_path, content in files:
        for chunk in chunk_text(content):
            record = {
                "text": f"File: {rel_path}\n\n{chunk}",
                "metadata": {"path": rel_path}
            }
            out.write(json.dumps(record, ensure_ascii=False) + "\n")
            pretrain_count += 1

print(f"Created pretraining dataset: {pretrain_path} ({pretrain_count} records)")

In [ ]:
# Chat SFT Dataset (messages format)
chat_path = os.path.join(OUTPUT_DIR, "chat_sft_dataset.jsonl")
chat_count = 0
with open(chat_path, "w", encoding="utf-8") as out:
    for rel_path, content in files:
        for chunk in chunk_text(content):
            record = {
                "messages": [
                    {"role": "user", "content": f"Provide the contents of file `{rel_path}`."},
                    {"role": "assistant", "content": chunk}
                ],
                "metadata": {"path": rel_path}
            }
            out.write(json.dumps(record, ensure_ascii=False) + "\n")
            chat_count += 1

print(f"Created Chat SFT dataset: {chat_path} ({chat_count} records)")

In [ ]:
# Inspect generated output samples
with open(chat_path, "r", encoding="utf-8") as f:
    sample = json.loads(f.readline())
print("Sample Chat Record:")
print(json.dumps(sample, indent=2)[:500])